# Model basics

I moved this first model experiment into a plain Colab notebook so the whole run is visible from top to bottom. I am starting with the five checked development complaints rather than pretending this is already a training result.

The local M1 run loaded the 4B checkpoint but became too slow during generation, so I will use a Colab GPU for this smoke test.

## What I am checking

I want to see whether the candidate model can follow the JSON contract before I build the training loop. This notebook checks the official chat template, deterministic decoding, a 4-bit loading path, and the LoRA adapter wiring. It also records the small run in MLflow.

In [ ]:
# Colab setup. Select a GPU runtime before running this cell.
!pip -q install "transformers==5.10.1" "peft==0.20.0" bitsandbytes accelerate "mlflow==3.15.1"


In [ ]:
import json
import time
from importlib.metadata import version
from pathlib import Path

import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU in Colab."

print("torch:", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0))


torch: 2.11.0+cu128
gpu: Tesla T4


## Load the checked complaints

I kept the data read simple. The notebook works when the repository root is the current directory, and also when the notebook is opened from its own folder.

In [ ]:
data_paths = [
    Path("data/gold_examples.jsonl"),
    Path("../data/gold_examples.jsonl"),
]
data_path = next(path for path in data_paths if path.exists())

examples = [
    json.loads(line)
    for line in data_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]

assert len(examples) == 5
print("data file:", data_path)
print("checked complaints:", len(examples))
examples[0]


data file: data/gold_examples.jsonl
checked complaints: 5


{'case_id': 'civic-example-001', 'surface_id': 'civic-example-001-a', 'split': 'development', 'source_type': 'manual_seed', 'style': 'informal_english', 'complaint': 'Bus 724 did not arrive near Nehru Place yesterday evening. I waited for almost an hour and do not know where to report it.', 'gold': {'service_domain': 'public_transport', 'issue_type': 'delay_or_non_arrival', 'location': 'near Nehru Place', 'event_date_or_time': 'yesterday evening', 'amount_inr': None, 'service_identifier': 'bus 724', 'urgency': 'routine', 'missing_information': ['exact_location'], 'formal_summary': 'Bus 724 did not arrive near Nehru Place yesterday evening, causing a wait of almost one hour.'}}

## Build the prompt

The model should return one JSON object and should not invent missing facts. I use the same prompt for every complaint so the output comparison stays fair.

In [ ]:
def build_messages(complaint):
    return [
        {
            "role": "system",
            "content": (
                "You structure public-service complaints. Return exactly one JSON object "
                "with these fields: service_domain, issue_type, location, "
                "event_date_or_time, amount_inr, service_identifier, urgency, "
                "missing_information, and formal_summary. Use null for an absent scalar "
                "fact. Use only the allowed labels from the task schema. Do not guess "
                "facts. Do not output reasoning or commentary."
            ),
        },
        {"role": "user", "content": complaint},
    ]


build_messages(examples[0]["complaint"])


## Load Qwen in 4-bit mode

This is the part that needs the Colab GPU. The base model stays frozen for the adapter check. QLoRA stores the frozen weights in 4-bit NF4 form and adds small trainable LoRA matrices later.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "main"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    quantization_config=quantization_config,
    device_map="auto",
)
model.eval()

print("model:", MODEL_NAME)
print("revision:", MODEL_REVISION)
print("quantization: 4-bit NF4")


model: Qwen/Qwen3-4B-Instruct-2507
revision: main
quantization: 4-bit NF4


## Generate fixed development outputs

I am using greedy decoding here. The output text and latency are saved before I add the dummy adapter.

In [ ]:
def generate_one(complaint, max_new_tokens=192):
    template_args = {
        "tokenize": True,
        "add_generation_prompt": True,
        "return_dict": True,
        "return_tensors": "pt",
        "enable_thinking": False,
    }
    try:
        inputs = tokenizer.apply_chat_template(
            build_messages(complaint),
            **template_args,
        )
    except TypeError:
        template_args.pop("enable_thinking")
        inputs = tokenizer.apply_chat_template(
            build_messages(complaint),
            **template_args,
        )

    inputs = inputs.to(next(model.parameters()).device)
    started = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            do_sample=False,
            max_new_tokens=max_new_tokens,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_tokens = inputs["input_ids"].shape[-1]
    response = tokenizer.decode(
        output[0][prompt_tokens:],
        skip_special_tokens=True,
    ).strip()
    return response, time.perf_counter() - started


outputs = []
for row in examples:
    response, latency = generate_one(row["complaint"])
    outputs.append(
        {
            "case_id": row["case_id"],
            "response": response,
            "latency_seconds": round(latency, 4),
        }
    )

for item in outputs:
    print("\ncase:", item["case_id"])
    print("latency:", item["latency_seconds"], "seconds")
    print(item["response"])



case: civic-example-001
latency: 12.4311 seconds
{
  "service_domain": "public_transport",
  "issue_type": "delayed_or_missed_service",
  "location": "Nehru Place",
  "event_date_or_time": null,
  "amount_inr": null,
  "service_identifier": "724",
  "urgency": "medium",
  "missing_information": null,
  "formal_summary": "The bus service numbered 724 failed to arrive at Nehru Place yesterday evening. The passenger waited for nearly an hour without the bus appearing, and there is no known point of contact for reporting the issue."
}

case: civic-example-002
latency: 10.7765 seconds
{
  "service_domain": "water supply",
  "issue_type": "no water supply",
  "location": "Block C, Shalimar Bagh",
  "event_date_or_time": null,
  "amount_inr": null,
  "service_identifier": null,
  "urgency": "high",
  "missing_information": [],
  "formal_summary": "There has been no water supply in Block C, Shalimar Bagh since last night, and the taps remain dry this morning. The issue is ongoing and requires

## Attach a dummy LoRA adapter

This does not train anything yet. I am only checking that the base parameters are frozen and the intended adapter parameters are the trainable part.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    bias="none",
    task_type="CAUSAL_LM",
)

adapted_model = get_peft_model(model, lora_config)
trainable_parameters = sum(
    parameter.numel()
    for parameter in adapted_model.parameters()
    if parameter.requires_grad
)
total_parameters = sum(
    parameter.numel()
    for parameter in adapted_model.parameters()
)
base_trainable_parameters = sum(
    parameter.numel()
    for name, parameter in adapted_model.named_parameters()
    if parameter.requires_grad and "lora_" not in name
)

assert base_trainable_parameters == 0
print("trainable parameters:", trainable_parameters)
print("total parameters:", total_parameters)
print("trainable percent:", round(100 * trainable_parameters / total_parameters, 4))


trainable parameters: 33030144
total parameters: 2238840320
trainable percent: 1.4753


## Save the small experiment

I am keeping this MLflow store inside the Colab runtime. If the run is useful, I can download the output files with the notebook results.

In [ ]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import mlflow

metadata = {
    "model_name": MODEL_NAME,
    "model_revision": MODEL_REVISION,
    "device": "cuda",
    "quantization_mode": "4-bit NF4",
    "decoding": {"do_sample": False, "max_new_tokens": 192},
    "mean_latency_seconds": round(
        sum(item["latency_seconds"] for item in outputs) / len(outputs),
        4,
    ),
    "lora": {
        "rank": 16,
        "alpha": 32,
        "dropout": 0.05,
        "target_modules": "all-linear",
        "trainable_parameters": trainable_parameters,
        "total_parameters": total_parameters,
    },
    "package_versions": {
        name: version(name)
        for name in ("torch", "transformers", "peft", "mlflow")
    },
    "gpu_memory_mb": round(torch.cuda.max_memory_allocated() / 2**20, 2),
}

mlflow.set_tracking_uri("file:///content/mlruns")
mlflow.set_experiment("civicstruct-model-basics")
with mlflow.start_run(run_name="qwen3-4b-development") as run:
    mlflow.log_params(
        {
            "model_name": metadata["model_name"],
            "model_revision": metadata["model_revision"],
            "device": metadata["device"],
            "quantization_mode": metadata["quantization_mode"],
            "lora_rank": metadata["lora"]["rank"],
            "lora_alpha": metadata["lora"]["alpha"],
        }
    )
    mlflow.log_metrics(
        {
            "mean_latency_seconds": metadata["mean_latency_seconds"],
            "trainable_percent": round(
                100 * trainable_parameters / total_parameters,
                4,
            ),
            "gpu_memory_mb": metadata["gpu_memory_mb"],
        }
    )
    mlflow.log_text(json.dumps(metadata, indent=2), "run_metadata.json")
    mlflow.log_text(json.dumps(outputs, indent=2), "outputs.json")

metadata["mlflow_run_id"] = run.info.run_id
print(json.dumps(metadata, indent=2))



{
  "model_name": "Qwen/Qwen3-4B-Instruct-2507",
  "model_revision": "main",
  "device": "cuda",
  "quantization_mode": "4-bit NF4",
  "decoding": {
    "do_sample": false,
    "max_new_tokens": 192
  },
  "mean_latency_seconds": 11.2558,
  "lora": {
    "rank": 16,
    "alpha": 32,
    "dropout": 0.05,
    "target_modules": "all-linear",
    "trainable_parameters": 33030144,
    "total_parameters": 2238840320
  },
  "package_versions": {
    "torch": "2.11.0+cu128",
    "transformers": "5.10.1",
    "peft": "0.20.0",
    "mlflow": "3.15.1"
  },
  "gpu_memory_mb": 2687.87,
  "mlflow_run_id": "28f873075d91428ea72a6ab87394f99f"
}


## Notes

This notebook is only the first smoke test. It does not claim that QLoRA improves the task, and the five complaints are not enough for a meaningful model comparison. The first run returned JSON-shaped text, but I have not treated it as schema-valid yet. The next notebook work should parse these outputs with the shared validator and build the controlled training split before any fine-tuning run.